# Week 9: Graph Representations, BFS, DFS & Topological Sort

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Adjacency list/matrix, unweighted shortest paths, DFS timestamps, cycle detection, and Kahn's algorithm.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-09/labs/W09_Graph_Representations_BFS_DFS.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Adjacency List vs. Matrix, BFS Unweighted Shortest Paths & Components

An adjacency list uses $O(V + E)$ memory for sparse graphs. Breadth-first search (BFS) explores layers outward from `start`, guaranteeing minimum hop counts.

In [1]:
from collections import deque


class Graph:
    def __init__(self, directed=False):
        self.directed = directed
        self.adj = {}

    def add_edge(self, u, v):
        self.adj.setdefault(u, [])
        self.adj.setdefault(v, [])
        self.adj[u].append(v)
        if not self.directed:
            self.adj[v].append(u)


def bfs_shortest(g, start):
    dist = {start: 0}
    parent = {start: None}
    q = deque([start])
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in g.adj.get(u, []):
            if v not in dist:
                dist[v] = dist[u] + 1
                parent[v] = u
                q.append(v)
    return order, dist, parent


g = Graph(directed=False)
for u, v in [("StoneTown", "Fumba"), ("Fumba", "Bweleo"), ("StoneTown", "Mbweni"), ("Mbweni", "Bweleo"), ("Nungwi", "Kendwa")]:
    g.add_edge(u, v)

order, dist, parent = bfs_shortest(g, "StoneTown")
print("BFS order from StoneTown:", order)
print("Hop distance to Bweleo  :", dist["Bweleo"])
assert dist["Bweleo"] == 2 and "Nungwi" not in dist


BFS order from StoneTown: ['StoneTown', 'Fumba', 'Mbweni', 'Bweleo']
Hop distance to Bweleo  : 2


## 2. DFS Discovery/Finish Timestamps & Kahn's Topological Sort

In a directed acyclic graph (DAG), Kahn's algorithm repeatedly removes zero in-degree vertices in $O(V + E)$ time, whereas a back edge to a gray vertex in DFS certifies a cycle.

In [2]:
def topological_sort_kahn(g):
    indeg = {u: 0 for u in g.adj}
    for u in g.adj:
        for v in g.adj[u]:
            indeg[v] += 1
    q = deque([u for u, d in indeg.items() if d == 0])
    topo = []
    while q:
        u = q.popleft()
        topo.append(u)
        for v in g.adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    if len(topo) != len(g.adj):
        raise ValueError("cycle detected")
    return topo


dag = Graph(directed=True)
for u, v in [("Python", "Z5007"), ("DiscreteMath", "Z5007"), ("Z5007", "ML"), ("Z5007", "BigData"), ("ML", "Thesis"), ("BigData", "Thesis")]:
    dag.add_edge(u, v)

order = topological_sort_kahn(dag)
print("Valid prerequisite schedule:", " -> ".join(order))
pos = {u: i for i, u in enumerate(order)}
assert all(pos[u] < pos[v] for u in dag.adj for v in dag.adj[u])


Valid prerequisite schedule: Python -> DiscreteMath -> Z5007 -> ML -> BigData -> Thesis
